# African Logistics Intelligence — Phase 2

## Road Network & Network Distance Analysis

### Objective

Connect the validated delivery and depot locations to a real Lagos road network and calculate road-network distances between locations.

### Pipeline

Validated delivery points
→ OpenStreetMap road network
→ nearest network nodes
→ shortest paths
→ network distances
→ cost matrix

This notebook forms the geospatial routing foundation for the Phase 3 baseline dispatcher.


In [3]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
from pathlib import Path

print("OSMnx:", ox.__version__)
print("GeoPandas:", gpd.__version__)
print("NetworkX:", nx.__version__)

OSMnx: 2.1.1
GeoPandas: 1.2.0
NetworkX: 3.7


In [7]:
# Get the project root (one level above the notebooks directory)
# In Jupyter, __file__ is not defined, so we work around it
import os
if '__file__' in dir():
    BASE_DIR = Path(__file__).parent.parent
else:
    # In Jupyter: assume we're in notebooks/ or running from project root
    BASE_DIR = Path.cwd()
    if BASE_DIR.name == 'notebooks':
        BASE_DIR = BASE_DIR.parent

DELIVERIES_GPKG = BASE_DIR / "deliveries_utm.gpkg"
OLD_GPKG = BASE_DIR / "lagos_logistics.gpkg"

print("Deliveries File Exists", DELIVERIES_GPKG.exists())
print("Old GeoPackage exists:", OLD_GPKG.exists())

Deliveries File Exists True
Old GeoPackage exists: True


In [3]:
import pyogrio

layers = pyogrio.list_layers(OLD_GPKG)

print("Layers in GeoPackage:")

for layer in layers:
    print(layer)

Layers in GeoPackage:
['deliveries' 'Point']
['depot' 'Point']


In [4]:
import geopandas as gpd

deliveries = gpd.read_file(
    DELIVERIES_GPKG,
    layer="lagos",
    engine="pyogrio"
)

print("Deliveries CRS:", deliveries.crs)
print("Number of deliveries:", len(deliveries))

deliveries.head()

Deliveries CRS: EPSG:32631
Number of deliveries: 60


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min,geometry
0,D001,C001,Maryland,6.575974,3.365894,2.1,SMALL,NORMAL,14:00:00,18:00:00,15,POINT (540449.018 726889.466)
1,D002,C002,Surulere,6.502082,3.365384,5.2,MEDIUM,NORMAL,12:00:00,16:00:00,5,POINT (540398.57 718720.912)
2,D003,C003,Maryland,6.570127,3.365127,9.9,MEDIUM,NORMAL,12:00:00,16:00:00,10,POINT (540364.698 726243.037)
3,D004,C004,Victoria Island,6.440734,3.428039,4.2,SMALL,HIGH,10:00:00,14:00:00,7,POINT (547331.829 711944.496)
4,D005,C005,Yaba,6.512044,3.375040,5.7,MEDIUM,NORMAL,10:00:00,14:00:00,10,POINT (541465.381 719822.959)


In [5]:
pyogrio.list_layers(OLD_GPKG)

array([['deliveries', 'Point'],
       ['depot', 'Point']], dtype=object)

In [6]:
depot = gpd.read_file(
    OLD_GPKG,
    layer="depot",
    engine="pyogrio"
)

print("Depot CRS:", depot.crs)
print("Number of depots:", len(depot))

depot

Depot CRS: EPSG:4326
Number of depots: 1


,depot_id,depot_name,area,latitude,longitude,opening_time,closing_time,geometry
0,DEP001,Apapa Central Depot,Apapa,6.4483,3.359,07:00:00,20:00:00,POINT (3.359 6.4483)


In [7]:
deliveries_wgs84 = deliveries.to_crs("EPSG:4326")
depot_wgs84 = depot.to_crs("EPSG:4326")

print("Deliveries:", deliveries_wgs84.crs)
print("Depot:", depot_wgs84.crs)

Deliveries: EPSG:4326
Depot: EPSG:4326


In [8]:
print("First delivery:")
print(deliveries_wgs84.geometry.iloc[0])

print("\nDepot:")
print(depot_wgs84.geometry.iloc[0])

First delivery:
POINT (3.365894 6.575974000000001)

Depot:
POINT (3.359 6.4483)


In [9]:
G = ox.graph_from_place(
    "Lagos, Nigeria",
    network_type="drive",
    simplify=True
)

G_proj = ox.project_graph(G)

In [10]:
graph_crs = G_proj.graph["crs"]

print("OSMnx projected CRS:", graph_crs)

OSMnx projected CRS: EPSG:32631


### OSMnx Graph Workflow

- **graph_from_place**: Downloads the drivable road network for Lagos from OpenStreetMap.
  - `network_type="drive"` → only roads accessible by cars.
  - `simplify=True` → merges redundant nodes for a cleaner graph.
  - **Result**: `G` is a NetworkX graph with intersections as nodes and road segments as edges.

- **project_graph(G)**: Reprojects the graph from EPSG:4326 (lat/long degrees) into a projected CRS (usually UTM in meters).
  - This is essential for accurate distance and routing calculations.
  - **Result**: `G_proj` has coordinates in meters.

- **graph["crs"]**: Stores the CRS metadata of the graph.
  - Printing it confirms which projection OSMnx chose (e.g., `EPSG:32631` for Lagos).
  - Ensures you know whether distances are in degrees or meters.

In [11]:
deliveries_proj = deliveries_wgs84.to_crs(graph_crs)
depot_proj = depot_wgs84.to_crs(graph_crs)

print("Graph:", graph_crs)
print("Deliveries:", deliveries_proj.crs)
print("Depot:", depot_proj.crs)

Graph: EPSG:32631
Deliveries: EPSG:32631
Depot: EPSG:32631


In [12]:
delivery_nodes, snap_distances = ox.distance.nearest_nodes(
    G_proj,
    X=deliveries_proj.geometry.x,
    Y=deliveries_proj.geometry.y,
    return_dist=True
)

deliveries_proj["network_node"] = delivery_nodes
deliveries_proj["snap_distance_m"] = snap_distances

deliveries_proj[
    ["delivery_id", "network_node", "snap_distance_m"]
].head(10)

,delivery_id,network_node,snap_distance_m
0,D001,1496513379,38.030521
1,D002,1221559553,50.690637
2,D003,5602623326,26.188862
3,D004,3243545465,139.306692
4,D005,10962867116,24.654341
5,D006,5602679447,52.470928
6,D007,1060399813,11.693705
7,D008,5196725721,51.423215
8,D009,7340396486,208.003165
9,D010,1047196381,39.652267


In [13]:
import sys
import scipy
import osmnx as ox

print("Python:", sys.executable)
print("SciPy:", scipy.__version__)
print("OSMnx:", ox.__version__)

Python: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\.venv\Scripts\python.exe
SciPy: 1.18.1
OSMnx: 2.1.1


In [14]:
depot_node, depot_snap_distance = ox.distance.nearest_nodes(
    G_proj,
    X=float(depot_proj.geometry.x.iloc[0]),
    Y=float(depot_proj.geometry.y.iloc[0]),
    return_dist=True
)

print("Depot network node:", depot_node)
print(f"Depot snap distance: {depot_snap_distance:.2f} m")

Depot network node: 1655215155
Depot snap distance: 26.74 m


Why we have two variables in the above code similar to deliveries_proj, is because we asked OSMnx to return both the nearest node and the distance to that node. We said return_dist=True, so it returns both the node ID and the distance in meters. This is useful for understanding how far each delivery point is from the road network.

In [15]:
deliveries_proj[
    ["delivery_id", "network_node", "snap_distance_m"]
].sort_values(
    "snap_distance_m",
    ascending=False
).head(10)

,delivery_id,network_node,snap_distance_m
24,D025,4395393082,742.927670
32,D033,2251308608,385.278088
56,D057,5390230451,302.512183
8,D009,7340396486,208.003165
55,D056,5581639698,170.778078
51,D052,1300349950,166.707217
48,D049,1472672752,144.257903
3,D004,3243545465,139.306692
52,D053,54591706,101.170456
43,D044,3740115953,97.428611


In [16]:
d025 = deliveries_proj[
    deliveries_proj["delivery_id"] == "D025"
]

edge, edge_dist = ox.distance.nearest_edges(
    G_proj,
    X=d025.geometry.x.iloc[0],
    Y=d025.geometry.y.iloc[0],
    return_dist=True
)

print("Nearest edge:", edge)
print(f"Distance to nearest routable edge: {edge_dist:.2f} m")

Nearest edge: (5603659683, 4395393082, 0)
Distance to nearest routable edge: 742.93 m


In [18]:
# Build mapping of location ID -> road-network node
location_nodes = {
    "DEP001": depot_node
}

location_nodes.update(
    dict(
        zip(
            deliveries_proj["delivery_id"],
            deliveries_proj["network_node"]
        )
    )
)

location_ids = list(location_nodes.keys())

# Empty distance matrix
distance_matrix = pd.DataFrame(
    index=location_ids,
    columns=location_ids,
    dtype=float
)

# Calculate shortest road-network distance
for origin_id, origin_node in location_nodes.items():
    for destination_id, destination_node in location_nodes.items():

        if origin_id == destination_id:
            distance_matrix.loc[origin_id, destination_id] = 0.0
            continue

        try:
            distance = nx.shortest_path_length(
                G_proj,
                source=origin_node,
                target=destination_node,
                weight="length"
            )

            distance_matrix.loc[
                origin_id,
                destination_id
            ] = distance

        except nx.NetworkXNoPath:
            distance_matrix.loc[
                origin_id,
                destination_id
            ] = float("nan")

distance_matrix.head()

KeyboardInterrupt: 

The code above does some certain things. We first create a dictionary of location and their respective nodes. Then we get the delivery id and delivery node nearest to each delivery point and add them to the dictionary using the update method. We then create an empty distance matrix with the location ids as both the index and columns.

### This is the simplest part
We make use of a for loop to iterate through each origin and destination pair. If the origin and destination are the same, we set the distance to 0. Otherwise, we use NetworkX's shortest_path_length function to calculate the shortest path length between the two nodes, using the "length" attribute of the edges as weights. If there is no path between two nodes, we catch the NetworkXNoPath exception and set the distance to NaN. Finally, we display the first few rows of the distance matrix.

In [ ]:
print("Matrix shape:", distance_matrix.shape)
print("Missing routes:", distance_matrix.isna().sum().sum())

distance_matrix.iloc[:10, :10]

In [ ]:
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "network_distance_matrix.csv"

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

distance_matrix.to_csv(OUTPUT_PATH)

print(f"Saved to: {OUTPUT_PATH}")

# Phase 3 — Rule-Based Dispatch Baseline

## Objective

Build a deliberately simple dispatch baseline for the Lagos last-mile delivery scenario.

The baseline will:

1. Filter unavailable riders.
2. Assign deliveries using simple operational rules.
3. Respect basic rider capacity constraints.
4. Sequence assigned deliveries using greedy nearest-next-stop routing.
5. Use the Phase 2 road-network distance matrix instead of straight-line distance.
6. Simulate each rider's route.
7. Measure baseline operational performance.

This baseline is not intended to be optimal.

It establishes a measurable benchmark that later optimization and ML components must improve upon.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Get the project root (one level above the notebooks directory)
# In Jupyter, __file__ is not defined, so we work around it
import os
if '__file__' in dir():
    BASE_DIR = Path(__file__).parent.parent
else:
    # In Jupyter: assume we're in notebooks/ or running from project root
    BASE_DIR = Path.cwd()
    if BASE_DIR.name == 'notebooks':
        BASE_DIR = BASE_DIR.parent

DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

deliveries_raw = pd.read_csv(DATA_RAW / "deliveries.csv")
riders = pd.read_csv(DATA_RAW / "riders.csv")

distance_matrix = pd.read_csv(
    DATA_PROCESSED / "network_distance_matrix.csv",
    index_col=0
)

print("Deliveries:", deliveries_raw.shape)
print("Riders:", riders.shape)
print("Distance matrix:", distance_matrix.shape)

Deliveries: (60, 11)
Riders: (6, 9)
Distance matrix: (61, 61)


In [2]:
print("DELIVERY COLUMNS")
print(deliveries_raw.columns.tolist())

print("\nRIDER COLUMNS")
print(riders.columns.tolist())

display(deliveries_raw.head())
display(riders)

DELIVERY COLUMNS
['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min']

RIDER COLUMNS
['rider_id', 'vehicle_type', 'available', 'shift_start', 'shift_end', 'break_start', 'break_end', 'capacity_kg', 'max_packages']


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min
0,D001,C001,Maryland,6.575974,3.365894,2.1,SMALL,NORMAL,14:00,18:00,15
1,D002,C002,Surulere,6.502082,3.365384,5.2,MEDIUM,NORMAL,12:00,16:00,5
2,D003,C003,Maryland,6.570127,3.365127,9.9,MEDIUM,NORMAL,12:00,16:00,10
3,D004,C004,Victoria Island,6.440734,3.428039,4.2,SMALL,HIGH,10:00,14:00,7
4,D005,C005,Yaba,6.512044,3.375040,5.7,MEDIUM,NORMAL,10:00,14:00,10


,rider_id,vehicle_type,available,shift_start,shift_end,break_start,break_end,capacity_kg,max_packages
0,R001,motorcycle,True,08:00,18:00,13:00,14:00,100,15
1,R002,motorcycle,True,08:00,18:00,13:00,14:00,100,15
2,R003,motorcycle,True,08:00,16:00,12:00,13:00,80,12
3,R004,motorcycle,True,10:00,19:00,14:00,15:00,100,15
4,R005,motorcycle,True,09:00,18:00,13:00,14:00,80,12
5,R006,motorcycle,False,08:00,18:00,12:00,13:00,100,15


In [3]:
deliveries = deliveries_raw.copy()
riders_work = riders.copy()

In [4]:
def time_to_minutes(time_value):
    """
    Convert HH:MM into minutes after midnight.

    Example:
    08:30 -> 510
    """
    hours, minutes = map(
        int,
        str(time_value).split(":")
    )

    return hours * 60 + minutes

In [5]:
available_riders = riders_work[
    riders_work["available"] == True
].copy()

print(
    f"Available riders: "
    f"{len(available_riders)} / {len(riders_work)}"
)

display(
    available_riders[
        [
            "rider_id",
            "shift_start",
            "shift_end",
            "capacity_kg",
            "max_packages"
        ]
    ]
)

Available riders: 5 / 6


,rider_id,shift_start,shift_end,capacity_kg,max_packages
0,R001,08:00,18:00,100,15
1,R002,08:00,18:00,100,15
2,R003,08:00,16:00,80,12
3,R004,10:00,19:00,100,15
4,R005,09:00,18:00,80,12


In [6]:
print(
    deliveries["priority"].value_counts(
        dropna=False
    )
)

display(
    deliveries[
        [
            "delivery_id",
            "area",
            "priority",
            "window_start",
            "window_end"
        ]
    ].head(15)
)

priority
NORMAL    28
HIGH      20
LOW       10
URGENT     2
Name: count, dtype: int64


,delivery_id,area,priority,window_start,window_end
0,D001,Maryland,NORMAL,14:00,18:00
1,D002,Surulere,NORMAL,12:00,16:00
2,D003,Maryland,NORMAL,12:00,16:00
3,D004,Victoria Island,HIGH,10:00,14:00
4,D005,Yaba,NORMAL,10:00,14:00
5,D006,Maryland,NORMAL,09:00,18:00
6,D007,Apapa,HIGH,10:00,14:00
7,D008,Lagos Island,NORMAL,09:00,12:00
8,D009,Lekki,HIGH,09:00,12:00
9,D010,Yaba,NORMAL,14:00,18:00


In [7]:
print("DELIVERY COLUMNS:")
print(deliveries.columns.tolist())

print("\nRIDER COLUMNS:")
print(riders_work.columns.tolist())

DELIVERY COLUMNS:
['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min']

RIDER COLUMNS:
['rider_id', 'vehicle_type', 'available', 'shift_start', 'shift_end', 'break_start', 'break_end', 'capacity_kg', 'max_packages']


In [8]:
priority_rank = {
    "urgent": 0,
    "high": 1,
    "medium": 2,
    "normal": 3,
    "low": 4
}

deliveries["priority_rank"] = (
    deliveries["priority"]
    .str.lower()
    .map(priority_rank)
    .fillna(99)
)

deliveries = deliveries.sort_values(
    [
        "window_end",
        "priority_rank",
        "delivery_id"
    ]
).reset_index(drop=True)

In [9]:
def build_greedy_route(
    delivery_ids,
    distance_matrix,
    depot_id="DEP001"
):
    """
    Build a simple nearest-neighbour route.

    Starts at the depot, repeatedly visits the nearest
    unvisited assigned delivery, then returns to the depot.
    """

    remaining = set(delivery_ids)

    route = [depot_id]
    current = depot_id
    total_distance_m = 0.0

    while remaining:

        next_delivery = min(
            remaining,
            key=lambda delivery_id:
                distance_matrix.loc[
                    current,
                    delivery_id
                ]
        )

        leg_distance = distance_matrix.loc[
            current,
            next_delivery
        ]

        total_distance_m += leg_distance

        route.append(next_delivery)

        remaining.remove(next_delivery)

        current = next_delivery

    if len(route) > 1:

        return_distance = distance_matrix.loc[
            current,
            depot_id
        ]

        total_distance_m += return_distance

        route.append(depot_id)

    return route, total_distance_m

In [10]:
test_deliveries = [
    "D001",
    "D002",
    "D003",
    "D004"
]

test_route, test_distance = build_greedy_route(
    test_deliveries,
    distance_matrix
)

print("Route:")
print(" → ".join(test_route))

print(
    f"\nTotal network distance: "
    f"{test_distance / 1000:.2f} km"
)

Route:
DEP001 → D002 → D003 → D001 → D004 → DEP001

Total network distance: 52.76 km


In [29]:
rider_state = {}

for _, rider in available_riders.iterrows():

    rider_id = rider["rider_id"]

    rider_state[rider_id] = {
        "deliveries": [],
        "total_weight_kg": 0.0,
        "package_count": 0,
        "areas": set()
    }

rider_state

{'R001': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R002': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R003': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R004': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()},
 'R005': {'deliveries': [],
  'total_weight_kg': 0.0,
  'package_count': 0,
  'areas': set()}}

In [31]:
assignments = []
unassigned = []

for _, delivery in deliveries.iterrows():

    delivery_id = delivery["delivery_id"]
    delivery_weight = float(delivery["weight_kg"])
    delivery_area = delivery["area"]

    feasible_riders = []

    for _, rider in available_riders.iterrows():

        rider_id = rider["rider_id"]
        state = rider_state[rider_id]

        # Check weight capacity
        new_weight = (
            state["total_weight_kg"]
            + delivery_weight
        )

        if new_weight > rider["capacity_kg"]:
            continue

        # Check package capacity
        new_package_count = (
            state["package_count"] + 1
        )

        if new_package_count > rider["max_packages"]:
            continue

        # Basic delivery-window / shift compatibility
        window_overlaps_shift = (
            delivery["window_end"]
            >= rider["shift_start"]
            and
            delivery["window_start"]
            <= rider["shift_end"]
        )

        if not window_overlaps_shift:
            continue

        # Prefer riders already serving this area
        same_area_penalty = (
            0
            if delivery_area in state["areas"]
            else 1
        )

        # Prefer rider with lower package utilization
        load_ratio = (
            new_package_count
            / rider["max_packages"]
        )

        score = (
            same_area_penalty,
            load_ratio
        )

        feasible_riders.append(
            (score, rider_id)
        )

    if not feasible_riders:

        unassigned.append(delivery_id)
        continue

    feasible_riders.sort(
        key=lambda x: x[0]
    )

    _, selected_rider = feasible_riders[0]

    state = rider_state[selected_rider]

    state["deliveries"].append(
        delivery_id
    )

    state["total_weight_kg"] += (
        delivery_weight
    )

    state["package_count"] += 1

    state["areas"].add(
        delivery_area
    )

    assignments.append(
        {
            "delivery_id": delivery_id,
            "rider_id": selected_rider,
            "area": delivery_area,
            "weight_kg": delivery_weight
        }
    )

SyntaxError: invalid syntax (3067352683.py, line 102)

In [ ]:
baseline_routes = []

for rider_id, state in rider_state.items():

    assigned_deliveries = (
        state["deliveries"]
    )

    if not assigned_deliveries:
        continue

    route, total_distance_m = (
        build_greedy_route(
            assigned_deliveries,
            distance_matrix,
            depot_id="DEP001"
        )
    )

    baseline_routes.append(
        {
            "rider_id": rider_id,
            "route": route,
            "delivery_count": len(
                assigned_deliveries
            ),
            "total_distance_m": (
                total_distance_m
            ),
            "total_distance_km": (
                total_distance_m / 1000
            )
        }
    )